# SVMRouter - Training

This notebook demonstrates how to train the **SVMRouter** (Support Vector Machine Router).

## Overview

SVMRouter uses a Support Vector Machine classifier to route queries to the most suitable LLM based on:
- Query embeddings (using Longformer)
- Historical performance data

**Key Features**:
- Effective in high-dimensional spaces
- Works well with clear margin of separation
- Supports probability estimation

## 1. Environment Setup

In [ ]:
# Install required packages (for Colab)
!pip install llmrouter-lib scikit-learn transformers torch
!git clone https://github.com/ulab-uiuc/LLMRouter.git
%cd LLMRouter

In [ ]:
import os
os.environ['OPENAI_API_KEY'] = 'your-key'
os.environ['ANTHROPIC_API_KEY'] = 'your-key'
# Or for multiple keys:
os.environ['API_KEYS'] = '["key1", "key2"]'

In [ ]:
# Import required modules
from llmrouter.models.svmrouter import SVMRouter, SVMRouterTrainer
from llmrouter.utils import setup_environment

setup_environment()
print("Environment setup complete!")

## 2. Configuration

SVMRouter uses the following configuration parameters:

| Parameter | Description | Default |
|-----------|-------------|--------|
| `kernel` | Kernel type: "rbf", "linear", "poly", "sigmoid" | "rbf" |
| `C` | Regularization parameter | 1.0 |
| `gamma` | Kernel coefficient | "scale" |
| `probability` | Enable probability estimates | true |

In [ ]:
import yaml

# Configuration file path
CONFIG_PATH = "configs/model_config_train/svmrouter.yaml"

# Load and display configuration
with open(CONFIG_PATH, 'r') as f:
    config = yaml.safe_load(f)

print("Current Configuration:")
print("=" * 50)
print(yaml.dump(config, default_flow_style=False))

## 3. Initialize Router

In [ ]:
# Initialize SVMRouter with configuration
router = SVMRouter(yaml_path=CONFIG_PATH)

print("Router initialized successfully!")
print(f"Number of training samples: {len(router.routing_data_train)}")
print(f"Number of LLM candidates: {len(router.llm_data)}")
print(f"LLM candidates: {list(router.llm_data.keys())}")

In [ ]:
# Inspect the SVM model configuration
print("SVM Model Parameters:")
print(router.svm_model.get_params())

## 4. Training

In [ ]:
# Initialize trainer
trainer = SVMRouterTrainer(router=router, device='cpu')

print("Trainer initialized!")
print(f"Training samples: {len(trainer.query_embedding_list)}")
print(f"Save path: {trainer.save_model_path}")

In [ ]:
# Train the model
print("Starting training...")
print("=" * 50)

trainer.train()

print("=" * 50)
print("Training completed!")

## 5. Model Verification

In [ ]:
# Verify the trained model
from llmrouter.utils import load_model
import numpy as np

# Load the saved model
saved_model = load_model(trainer.save_model_path)

print("Model loaded successfully!")
print(f"Model type: {type(saved_model).__name__}")
print(f"Number of support vectors: {saved_model.n_support_}")
print(f"Classes: {saved_model.classes_}")

In [ ]:
# Quick prediction test
test_embedding = trainer.query_embedding_list[0].reshape(1, -1)
prediction = saved_model.predict(test_embedding)

print(f"Test prediction: {prediction[0]}")

# Get prediction probabilities
proba = saved_model.predict_proba(test_embedding)
print(f"\nPrediction probabilities:")
for model, prob in zip(saved_model.classes_, proba[0]):
    print(f"  {model}: {prob:.4f}")

## 6. Hyperparameter Tuning

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC
import numpy as np

# Prepare data
X = np.array(trainer.query_embedding_list)
y = np.array(trainer.model_name_list)

# Define parameter grid
param_grid = {
    'C': [0.1, 1, 10],
    'kernel': ['rbf', 'linear'],
    'gamma': ['scale', 'auto']
}

print("Running grid search (this may take a while)...")

# Grid search
svm = SVC(probability=True)
grid_search = GridSearchCV(svm, param_grid, cv=3, scoring='accuracy', verbose=1, n_jobs=-1)
grid_search.fit(X, y)

print(f"\nBest parameters: {grid_search.best_params_}")
print(f"Best score: {grid_search.best_score_:.4f}")

## Summary

In this notebook, we:

1. **Loaded Configuration**: Set up SVMRouter with YAML configuration
2. **Trained Model**: Used SVMRouterTrainer to fit the SVM classifier
3. **Verified Model**: Loaded and tested the saved model
4. **Tuned Hyperparameters**: Found optimal parameters using grid search

**Next Steps**:
- Use the next part of notebook for inference
- Compare with other routers (KNN, MLP)

# SVMRouter - Inference

This part of notebook demonstrates how to use a trained **SVMRouter** for inference.

## 1. Environment Setup (optional)

In [ ]:
# Install required packages (for Colab)
!pip install llmrouter-lib transformers torch
!git clone https://github.com/ulab-uiuc/LLMRouter.git
%cd LLMRouter

In [ ]:
import os
os.environ['OPENAI_API_KEY'] = 'your-key'
os.environ['ANTHROPIC_API_KEY'] = 'your-key'
# Or for multiple keys:
os.environ['API_KEYS'] = '["key1", "key2"]'

In [ ]:
from llmrouter.models.svmrouter import SVMRouter
from llmrouter.utils import setup_environment, load_model, get_longformer_embedding
import yaml

setup_environment()

## 2. Load Trained Router

In [ ]:
# Configuration
CONFIG_PATH = "configs/model_config_train/svmrouter.yaml"

with open(CONFIG_PATH, 'r') as f:
    config = yaml.safe_load(f)

config['model_path']['load_model_path'] = config['model_path']['save_model_path']

# Create inference config
INFERENCE_CONFIG_PATH = "configs/model_config_test/svmrouter_inference.yaml"
os.makedirs(os.path.dirname(INFERENCE_CONFIG_PATH), exist_ok=True)

with open(INFERENCE_CONFIG_PATH, 'w') as f:
    yaml.dump(config, f)

# Initialize router
router = SVMRouter(yaml_path=INFERENCE_CONFIG_PATH)
print(f"Router loaded with {len(router.llm_data)} LLM candidates")

## 3. Single Query Routing

In [ ]:
# Example queries
EXAMPLE_QUERIES = [
    {"query": "What is the capital of France?"},
    {"query": "Solve the equation: 2x + 5 = 15"},
    {"query": "Write a Python function to check if a number is prime."},
    {"query": "Explain quantum computing in simple terms."},
]

print("Routing Results:")
print("=" * 60)

for i, query in enumerate(EXAMPLE_QUERIES, 1):
    result = router.route_single(query)
    print(f"{i}. {query['query'][:50]}...")
    print(f"   Routed to: {result['model_name']}")

## 4. Routing Probabilities

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path
PROJECT_ROOT = Path(os.getcwd()).parent.parent
project_root = os.getcwd()
# Load trained model
model_path = os.path.join(PROJECT_ROOT, config['model_path']['load_model_path'])

try:
    svm_model = load_model(model_path)
except Exception:
    model_path_alt = os.path.join(
        project_root,
        "llmrouter",
        config['model_path']['load_model_path']
    )
    svm_model = load_model(model_path_alt)
def get_routing_probabilities(query_text):
    embedding = get_longformer_embedding(query_text).numpy().reshape(1, -1)
    proba = svm_model.predict_proba(embedding)[0]
    return dict(sorted(zip(svm_model.classes_, proba), key=lambda x: x[1], reverse=True))
# Visualize for first query
query = EXAMPLE_QUERIES[0]['query']
probs = get_routing_probabilities(query)

plt.figure(figsize=(10, 5))
plt.barh(list(probs.keys()), list(probs.values()))
plt.xlabel('Probability')
plt.title(f'Routing Probabilities for: "{query[:40]}..."')
plt.tight_layout()
plt.show()

## 5. Batch Routing

## 6. File-Based Inference

Load queries from a file and save results.

In [ ]:
import json

# Load queries from a JSONL file
def load_queries_from_file(file_path):
    """Load queries from a JSONL file."""
    queries = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                queries.append(json.loads(line))
    return queries

# Save results to a JSONL file
def save_results_to_file(results, output_path):
    """Save routing results to a JSONL file."""
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, 'w', encoding='utf-8') as f:
        for result in results:
            f.write(json.dumps(result, ensure_ascii=False) + '\n')
    print(f"Results saved to: {output_path}")

# Example: Load from default query file
QUERY_FILE = "data/example_data/query_data/default_query_test.jsonl"
OUTPUT_FILE = "outputs/svmrouter_results.jsonl"

if os.path.exists(QUERY_FILE):
    # Load queries
    file_queries = load_queries_from_file(QUERY_FILE)
    print(f"Loaded {len(file_queries)} queries from: {QUERY_FILE}")
    
    # Route queries (using route_batch for efficiency)
    file_results = router.route_batch(batch=file_queries[:10])
    print(f"Routed {len(file_results)} queries")
    
    # Save results
    save_results_to_file(file_results, OUTPUT_FILE)
    
    # Show sample results
    print(f"\nSample results:")
    for i, result in enumerate(file_results[:3], 1):
        print(f"  {i}. {result.get('query', '')[:40]}... -> {result['model_name']}")
else:
    print(f"Query file not found: {QUERY_FILE}")
    print("Create a JSONL file with format: {\"query\": \"Your question\"}")

In [ ]:
from collections import Counter

# Route multiple queries
results = [router.route_single(q) for q in EXAMPLE_QUERIES]

# Show distribution
model_counts = Counter(r['model_name'] for r in results)
print("Routing Distribution:")
for model, count in model_counts.most_common():
    print(f"  {model}: {count}")

## Summary

This notebook demonstrated:
1. Loading a trained SVMRouter
2. Single and batch query routing
3. Visualizing routing probabilities

SVMRouter is effective for:
- High-dimensional embedding spaces
- Clear separation between LLM performance profiles